In [0]:
# Cell 0: Install any missing packages if needed
# (PyYAML no longer needed since we're using JSON)
# %pip install ...

# Cell 1: Imports
from pyspark.sql import SparkSession
from curated_config_loader import load_curated_config
from save_table import save_as_curated_table


In [0]:
# Cell 3: Load curated table config from JSON
config_path = "/Workspace/Users/soumyamukherjee42@gmail.com/personal_development/Curated/curated_tables.json"
curated_table_defs = load_curated_config(config_path)

In [0]:
from pyspark.sql import functions as F

for table_def in curated_table_defs:
    source_table = table_def["source_table"]
    table_name = table_def["table_name"]
    columns = table_def["columns"]
    dedup_col = table_def.get("deduplicate_on")
    join_info = table_def.get("join_with")

    print(f"\n📦 Creating curated table: curated_{table_name} from {source_table}")

    # Load source table
    df = spark.table(source_table)

    # Join if defined
    if join_info:
        join_df = spark.table(join_info["table"])
        df = df.join(join_df, on=join_info["on"], how=join_info.get("how", "inner"))

    # Build list of Column objects
    select_exprs = []

    for col in columns:
        if isinstance(col, str):
            select_exprs.append(F.col(col))
        elif isinstance(col, dict):
            if "expr" in col and "alias" in col:
                select_exprs.append(F.expr(col["expr"]).alias(col["alias"]))
            elif "alias" in col and "from_table" in col:
                select_exprs.append(F.col(f"{col['from_table']}.{col['alias']}").alias(col["alias"]))
            else:
                raise ValueError(f"Invalid column definition: {col}")
        else:
            raise ValueError(f"Unsupported column type: {col}")

    df = df.select(*select_exprs)

    # Deduplicate
    if dedup_col:
        df = df.dropDuplicates([dedup_col])

    # Save
    save_as_curated_table(df, table_name)
